**MÓDULO 04 - 1.4_E - PROJETO - FAZENDO A INGESTÃO NA CAMADA BRONZE**

1. Depois do nosso repositório criado no GitHub vamos agora trabalhar aqui no VSCode junto com o GitHub.
2. Vamos começar a ingestão dessas tabelas.
2. Vou salvar esse projeto na minha pasta de scripts do projeto: C:\Users\arion\desenvolvimento\myproject\scripts

In [1]:
# Para começar o nosso trabalho, vamos importar a biblioteca duckdb;
# Importar o pandas para trabalhar com os DataFrames
# Importar a biblioteca os para trabalhar com arquivos e diretórios
# Importar a biblioteca datetime para trabalhar com datas e horas
# Observação (erro: ModuleNotFoundError: No module named 'duckdb'):
# No VSCode/Jupyter, o pacote precisa estar instalado no MESMO kernel selecionado.
# Para instalar no kernel atual, execute em um terminal:
#   python -m pip install duckdb
import duckdb
import pandas as pd
import os
from datetime import datetime

In [19]:
# Isso que fez resolver o meu erro que estava dando para instalar no terminal e importar a bilbioteca
!pip install duckdb
!pip install pandas


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\arion\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\arion\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [2]:
# Depois de importar as bibliotecas, vamos criar a conexão com o banco de dados.
# Além de criar a conexão estou pedindo para ele salvar em memória e criar um arquivo chamado dados_duckdb.db, caso ele não exista. 
# O parâmetro read_only=False é para permitir que possamos escrever no banco de dados.

con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [3]:
# Agora vamos fazer a leitura dos arquivos CSV que estão na pasta landing.
# Agora vamos criar um dataframe e estou informando deste arquivo aqui, no diretório.../landing/z0019_1.csv')
df = pd.read_csv('../landing/z0019_1.csv')

In [4]:
# Vamos ver como ele leu esse arquivo z0019_1.csv
df.head()
# aqui ele leu como uma coluna só, então vamos informar para ele que o separador é o ponto e vírgula ';'
# Então vamos informar para ele que o separador é o ponto e vírgula ';'
df = pd.read_csv('../landing/z0019_1.csv', sep=';')

In [5]:
# Vamos ver como ficou agora o nosso dataframe, depois de informar o separador para ele.
df.head()

,NATBR,MAKTX,WERKS,MAINS,LABST
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT50,100,500
2,10003,PREGO,BT10,100,50


In [7]:
# Agora eu vou seprar algumas informações que sao o nome do arquivo, e no caminho eu vou utilizar como uma variável
# No caminho eu coloquei f' como parte de formatação do meu texto, e entre {} eu coloquei a variável 'arquivo'
# Então aqui ele vai conseguir ler o arquivo com base aqui no nome da variável 'arquivo' que eu criei, e o caminho é ../landing/{arquivo}
# vou colocar essa informação dentro de uma variável chamada data_ingestao
# Agora vou trazer essas informações como colunas.
# 'nome_arquivo' vai ser igual ao conteúdo da variável 'arquivo' e a mesma coisa para a data_ingestao, que vai ser igual ao conteúdo da variável 'data_ingestao'   
# Dessa forma eu consigo ter informações de quando foi realizada a ingestão e por qual arquivo a gente recebeu essas informações aqui.

arquivo = 'z0019_2.csv'
data_ingestao = datetime.now()
df = pd.read_csv(f'../landing/{arquivo}', sep=';')
df['nome_arquivo'] = arquivo
df['data_ingestao'] = data_ingestao
df.head()

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10004,SERRA,BT50,100,200,z0019_2.csv,2026-03-08 16:49:09.064132
1,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-03-08 16:49:09.064132
2,10003,PREGO,BT10,100,50,z0019_2.csv,2026-03-08 16:49:09.064132


In [6]:
# Agora eu quero a informação de data e hora de quando esse arquivo foi processado.
print(datetime.now())
# vou colocar essa informação dentro de uma variável chamada data_ingestao
data_ingestao = datetime.now()

2026-03-08 16:49:06.640573


In [8]:
# Agora podemos partir para a próxima etapa que é fazer a ingestão desses arquivos em uma TABELA BRONZE.
# Para isso, vamos utilizar o comando con.execute() para criar a tabela bronze e depois vamos utilizar o comando con.execute() para inserir os dados do nosso dataframe nessa tabela bronze.    
# Vamos colocar o nome das colunas conforme criamos

con.execute(query='''CREATE TABLE IF NOT EXISTS bronze_produtos(
                NATBR VARCHAR,
                MAKTX VARCHAR,  
                WERKS VARCHAR,
                MAINS VARCHAR,
                LABST VARCHAR,
                nome_arquivo VARCHAR,
                data_ingestao TIMESTAMP
            )
        ''')

In [9]:
# Agora iremos fazer a ingestão desses dados nessa tabela bronze_produtos, 
# utilizando o comando con.execute() para inserir os dados do nosso dataframe nessa tabela bronze.
# Lá no arquivo z0019_1.csv temos essas informações e vamos passar para a tabela bronze_produtos.
# Mas antes de fazer essa ingestão, vamos validar o conteúdo dessa tabela, pois no dataframe já vimos o conteúdo e validamos. 
con.execute('SELECT * FROM bronze_produtos').fetchdf()


,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10004,SERRA,BT50,100,200,z0019_2.csv,2026-03-05 22:56:31.796460
1,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-03-05 22:56:31.796460
2,10003,PREGO,BT10,100,50,z0019_2.csv,2026-03-05 22:56:31.796460
3,10004,SERRA,BT50,100,200,z0019_2.csv,2026-03-07 17:24:05.817779
4,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-03-07 17:24:05.817779
5,10003,PREGO,BT10,100,50,z0019_2.csv,2026-03-07 17:24:05.817779


In [14]:
# Vamos agora efetuar a ingestão desses dados nessa tabela bronze_produtos, utilizando o comando con.execute() para inserir os dados do nosso dataframe nessa tabela bronze.
# Vamos fazer a ingestão fazendo um select no dataframe e passando para a tabela bronze_produtos, utilizando o comando con.execute() para inserir os dados do nosso dataframe nessa tabela bronze. 
# Aqui eu estou selecionando tudo o que está nesse dataframe e fazendo o insert desses dados nessa tabela bronze_produtos.
con.execute('''
INSERT INTO bronze_produtos SELECT * FROM df
''')

In [11]:
# Verificando a ingestão dos dados, fazendo um select nessa tabela bronze_produtos para validar se os dados foram inseridos corretamente.
con.execute('SELECT * FROM bronze_produtos').fetchdf()

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10004,SERRA,BT50,100,200,z0019_2.csv,2026-03-05 22:56:31.796460
1,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-03-05 22:56:31.796460
2,10003,PREGO,BT10,100,50,z0019_2.csv,2026-03-05 22:56:31.796460
3,10004,SERRA,BT50,100,200,z0019_2.csv,2026-03-07 17:24:05.817779
4,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-03-07 17:24:05.817779
5,10003,PREGO,BT10,100,50,z0019_2.csv,2026-03-07 17:24:05.817779


In [13]:
# Podemos efetuar a consulta colocando dentro de uma variável Ex 'resultado' e depois utilizando o comando .head() para mostrar as primeiras linhas do resultado.
resultado = con.execute('SELECT * FROM bronze_produtos').fetchdf()
resultado.head()

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10004,SERRA,BT50,100,200,z0019_2.csv,2026-03-05 22:56:31.796460
1,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-03-05 22:56:31.796460
2,10003,PREGO,BT10,100,50,z0019_2.csv,2026-03-05 22:56:31.796460
3,10004,SERRA,BT50,100,200,z0019_2.csv,2026-03-07 17:24:05.817779
4,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-03-07 17:24:05.817779


Vamos fazer agora a mesma coisa para o arquivo z0019_2.csv, ou seja, vamos ler esse arquivo, criar um dataframe, validar o conteúdo do dataframe, e depois fazer a ingestão desses dados nessa tabela bronze_produtos.

1. Apenas vamos mudar o nome do arquivo de z0019_1.csv para z0019_2.csv
2. Executo o mesmo comando para criar a tabela bronze_produtos
3. Agora pode executar o comando acima de inserção sobre o dataframe para popular a tabela bronze_produtos sobre o z0019_2.csv
4. Agora pode executar o comando acima de consulta

→ Agora temos os registros dos dois arquivos z0019_1.csv e z0019_2.csv.
  Temos a data, horário e período no qual foi feita a ingestão, e também as colunas, conforme a gente leu, ou seja, são as tabelas BRUTAS e CRUAS, sem nenhuma alteração.

→ Esse foi um exemplo de como fazemos a ingestão na camada BRONZE
→ A gente vai receber esses arquivos, podemos colocar sim informações em metadados em colunas, como o nome do arquivo, data de ingestão, o usuário que disponibilizou, enfim, várias informações
→ Aqui na CAMADA BRONZE o data PERMANECE EXATAMENTE COMO ELE CHEGOU.
→ Agora vamos consumir esses dados na nossa CAMADA SILVER, E AÍ SIM, VAMOS EFETUADO OS AJUSTES e ENRIQUECIMENTO.

Salvando o arquivo, agora vamos finalizar a conexão

Mas antes tem um ponto interessante pra gente ver que é o seguite: 

→ Em alguns casos eu já posso trazer a tradução dessa tabela 'o nome das colunas sem sentido' para a CAMADA BRONZE.
→ Em outros casos a gente também vai encontrar a table 'produtos' apenas com esse nome 'bronze_produtos' na CAMADA SILVER.
→ Então também é possível a gente encontrar ela no seguinte formato...Vou atualizar a tabela 'bronze_produtos' vou renomeá-la.
→ Então aquim eu tenho a tambela também com o seu nome bruto.
→ Consigo e consegui identificar na origem exatamente como ela é chamada e identificada com o nome exato que ela tem na origem.

In [14]:
# Renomenando a tabela bronze_produtos para bronze_produtos_2024, utilizando o comando con.execute() para renomear a tabela.
con.execute('ALTER TABLE bronze_produtos RENAME TO bronze_z0019')

CatalogException: Catalog Error: Could not rename "bronze_produtos" to "bronze_z0019": another entry with this name already exists!

In [16]:
# Se eu executar o comando acima de consulta, vai dar erro que a tabela não existe.
# Mas executo com o nome da tabela atual agora que é bronze_z0019, ele vai mostrar os dados normalmente.
con.execute('SELECT * FROM bronze_z0019').fetchdf()

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-03-04 10:22:55.262189
1,10002,MARTELO,BT50,100,500,z0019_1.csv,2026-03-04 10:22:55.262189
2,10003,PREGO,BT10,100,50,z0019_1.csv,2026-03-04 10:22:55.262189
3,10004,SERRA,BT50,100,200,z0019_2.csv,2026-03-04 10:40:35.014454
4,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-03-04 10:40:35.014454
5,10003,PREGO,BT10,100,50,z0019_2.csv,2026-03-04 10:40:35.014454


In [17]:
# Fechando a conexão com o banco de dados
con.close()

**Pronto!! Banco atualizado, integrado e fechado na CAMADA BRONZE.**

**Partiu camada Silver!**